In [ ]:
import pandas as pd
import numpy as np
from pandas.tseries.frequencies import to_offset

from merlion.models.anomaly.autoencoder import AutoEncoder, AutoEncoderConfig
from merlion.models.anomaly.dagmm import DAGMM, DAGMMConfig
from merlion.transform.normalize import MeanVarNormalize
from merlion.post_process.threshold import AggregateAlarms
from merlion.evaluate.anomaly import TSADMetric
from merlion.utils import TimeSeries
from merlion.transform.base import Identity
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    precision_score,
    recall_score,
    roc_curve,
    roc_auc_score,
    f1_score,
    average_precision_score,
    precision_recall_fscore_support,
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
    auc
)
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import time
import json
import random
import torch
import torch.nn.functional as F
from merlion.models.anomaly import dagmm as dagmm_module
import optuna
import os
from openpyxl.utils import get_column_letter
from openpyxl import load_workbook

def set_seed(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    #torch.use_deterministic_algorithms(False)

    #torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = False

c:\Users\al.melnikova\AppData\Local\anaconda3\envs\py311cuda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def event_lengths(events):
    '''
    Вычисляет минимальную и медианную длительность событий в минутах.

    Параметры:
        events (list[list[str, str]]): список интервалов [[start, end], ...]

    Возвращает:
        dict: {
            'min': минимальная длительность,
            'median': медианная длительность
        }
    '''
    durations = []
    for start, end in events:
        start = pd.to_datetime(start)
        end = pd.to_datetime(end)

        duration = (end - start).total_seconds() / 60
        durations.append(duration)

    return {
        'min': min(durations),
        'median': np.median(durations)
    }

def event_params(min_event_len, median_event_len):
    '''
    Рассчитывает стартовые параметры для пайплайна на основе длительности событий.

    Параметры:
        min_event_len (float): минимальная длительность события (мин)
        median_event_len (float): медианная длительность события (мин)

    Возвращает:
        tuple[int, int, int, int]:
            (gap_fill, min_event, event_interval, window_size)
            — параметры, ограниченные заданными диапазонами
    '''

    def clip(value, vmin, vmax):
        return max(vmin, min(value, vmax))

    gap_fill = clip(min_event_len, 1, 15)
    min_event = clip(min_event_len * 0.5, 3, 120)
    event_interval = clip(median_event_len * 4 / 5, 20, 300)
    window_size = clip(median_event_len * 0.4, 5, 120)

    return int(gap_fill), int(min_event), int(event_interval), int(window_size)

def pr_auc(y_true, scores):
    '''
    PR-AUC / Average Precision
    '''
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    return average_precision_score(y_true, scores)

def binary_to_events(y_bin: pd.Series, gap_minutes=10, min_len_minutes=5):
    '''
    Преобразует бинарный временной ряд (0/1) в список событий (start, end).

    Объединяет точки со значением 1 в события, если разрыв между ними
    не превышает gap_minutes, и отбрасывает события короче min_len_minutes.

    Параметры:
        y_bin (pd.Series): временной ряд с индексом datetime и значениями 0/1
        gap_minutes (int): максимальный разрыв (в минутах) для объединения
        min_len_minutes (int): минимальная длительность события (в минутах)

    Возвращает:
        List[Tuple[pd.Timestamp, pd.Timestamp]]: список интервалов (start, end)
    '''
    y_bin = y_bin.sort_index().astype(int)
    ones = y_bin[y_bin == 1].dropna()
    if ones.empty:
        return []

    gap = pd.Timedelta(minutes=gap_minutes)
    min_len = pd.Timedelta(minutes=min_len_minutes)

    times = ones.index
    events = []
    start = prev = times[0]

    for t in times[1:]:
        if (t - prev) <= gap:
            prev = t
        else:
            end = prev
            if (end - start) >= min_len:
                events.append((start, end))
            start = prev = t

    end = prev
    if (end - start) >= min_len:
        events.append((start, end))

    return events

def f1_at_fpr(y_true, scores, target_fpr=0.01):
    '''
    Вычисляет F1-меру при ограничении на уровень ложноположительных срабатываний(FPR).

    Параметры:
        y_true (array-like): бинарные метки (0 — норма, 1 — аномалия)
        scores (array-like): anomaly score
        target_fpr (float): максимальный допустимый FPR

    Возвращает:
        float: значение F1 при пороге, дающем FPR ≤ target_fpr
    '''
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    fpr, tpr, thresholds = roc_curve(y_true, scores)

    valid_idx = np.where(fpr <= target_fpr)[0]
    if len(valid_idx) == 0:
        return 0.0

    idx = valid_idx[-1]
    threshold = thresholds[idx]

    y_pred = (scores >= threshold).astype(int)
    return f1_score(y_true, y_pred, zero_division=0)

def precision_at_k(y_true, scores, k):
    '''
    Вычисляет Precision@K — долю аномалий среди K наибольших score.

    Параметры:
        y_true (array-like): бинарные метки (0/1)
        scores (array-like): оценки аномальности (чем больше, тем выше ранг)
        k (int): количество топ-элементов

    Возвращает:
        float: Precision@K
    '''
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    if k <= 0:
        raise ValueError('k must be > 0')

    if len(y_true) == 0:
        return 0.0

    k = min(k, len(y_true))
    top_k_idx = np.argsort(scores)[::-1][:k]

    return float(y_true[top_k_idx].sum() / k)

def recall_at_k(y_true, scores, k):
    '''
    Вычисляет Recall@K — долю всех аномалий, попавших в топ-K по score.

    Параметры:
        y_true (array-like): бинарные метки (0/1)
        scores (array-like): оценки аномальности
        k (int): количество топ-элементов

    Возвращает:
        float: Recall@K
    '''
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    if k <= 0:
        raise ValueError('k must be > 0')

    total_anomalies = y_true.sum()
    if total_anomalies == 0:
        return 0.0

    k = min(k, len(y_true))
    top_k_idx = np.argsort(scores)[::-1][:k]
    found_anomalies = y_true[top_k_idx].sum()

    return float(found_anomalies / total_anomalies)

def _overlap(start1, end1, start2, end2):
    '''
    Проверяет, пересекаются ли два временных интервала.

    Параметры:
        start1, end1: границы первого интервала
        start2, end2: границы второго интервала

    Возвращает:
        bool: True, если интервалы пересекаются, иначе False
    '''
    return not (end1 < start2 or end2 < start1)

def _inside_true_event(pred_start, pred_end, true_start, true_end):
    '''
    Проверяет, находится ли предсказанное событие полностью внутри истинного.

    Параметры:
        pred_start, pred_end: границы предсказанного интервала
        true_start, true_end: границы истинного интервала

    Возвращает:
        bool: True, если предсказанное событие полностью внутри истинного
    '''
    return true_start <= pred_start and pred_end <= true_end

def _is_timely_detection(pred_start, true_start, max_min=180):
    '''
    Проверяет, является ли предсказание своевременным относительно истинного события.

    Параметры:
        pred_start: время начала предсказанного события
        true_start: время начала истинного события
        max_min (int): допустимое отклонение (в минутах)

    Возвращает:
        bool: True, если pred_start находится в окне вокруг true_start
    '''
    window = pd.Timedelta(minutes=max_min)
    return (true_start - window) <= pred_start <= (true_start + window)

def event_recall(true_events, pred_events, max_min=180):
    '''
    Вычисляет Event Recall — долю найденных истинных событий.

    Истинное событие считается найденным, если существует предсказанное событие,
    начало которого попадает в допустимый интервал до или после начала
    истинного события.

    Параметры:
        true_events (list[tuple]): истинные интервалы (start, end)
        pred_events (list[tuple]): предсказанные интервалы (start, end)
        max_min (int | float): допустимое отклонение по времени (мин)

    Возвращает:
        float: Event Recall
    '''
    if len(true_events) == 0:
        return 0.0

    detected = 0

    for t_start, t_end in true_events:
        found = False

        for p_start, p_end in pred_events:
            if _is_timely_detection(p_start, t_start, max_min=max_min):
                found = True
                break

        if found:
            detected += 1

    return detected / len(true_events)

def event_precision(true_events, pred_events, max_min=180):
    '''
    Вычисляет Event Precision — долю корректных предсказанных событий.

    Предсказанное событие считается корректным, если:
    1) его начало попадает в допустимый интервал вокруг начала истинного события, или
    2) оно полностью находится внутри истинного события.

    Параметры:
        true_events (list[tuple]): истинные интервалы (start, end)
        pred_events (list[tuple]): предсказанные интервалы (start, end)
        max_min (int | float): допустимое отклонение по времени (мин)

    Возвращает:
        float: Event Precision
    '''
    if len(pred_events) == 0:
        return 0.0

    correct_pred_idx = set()

    for t_start, t_end in true_events:
        for i, (p_start, p_end) in enumerate(pred_events):
            if _is_timely_detection(p_start, t_start, max_min=max_min):
                correct_pred_idx.add(i)

            elif _inside_true_event(p_start, p_end, t_start, t_end):
                correct_pred_idx.add(i)

    return len(correct_pred_idx) / len(pred_events)

def time_to_detect(true_events, pred_events, reduction='mean', unit='minutes', max_min=180, absolute=False):
    window = pd.Timedelta(minutes=max_min)
    delays = []

    for t_start, t_end in true_events:
        valid_starts = []

        for p_start, p_end in pred_events:
            if (t_start - window) <= p_start <= (t_start + window):
                valid_starts.append(p_start)

        if valid_starts:
            first_detection = min(valid_starts)
            delay = first_detection - t_start
            delays.append(_convert_unit(delay, unit))

    if not delays:
        return np.nan

    values = [abs(i) for i in delays] if absolute else delays

    if reduction == 'list':
        return values
    if reduction == 'mean':
        return float(np.mean(values))
    if reduction == 'median':
        return float(np.median(values))

    raise ValueError('reduction must be mean, median, or list')

def _convert_unit(td, unit):
    '''
    Конвертирует timedelta в заданные единицы времени.

    Параметры:
        td (pd.Timedelta): временной интервал
        unit (str): единицы ('seconds', 'minutes', 'hours')

    Возвращает:
        float: значение в выбранных единицах
    '''
    seconds = td.total_seconds()

    if unit == 'seconds':
        return seconds
    if unit == 'minutes':
        return seconds / 60.0
    if unit == 'hours':
        return seconds / 3600.0

    raise ValueError('unit must be seconds, minutes, or hours')

def alerts_per_day(pred_events, timestamps, gap_threshold=None):
    """
    Среднее количество алертов в день с учетом разрывов во временном ряду.

    Parameters
    ----------
    pred_events : list[tuple]
        Предсказанные интервалы (start, end)
    timestamps : pd.DatetimeIndex | list-like
        Временные метки наблюдений
    gap_threshold : str | pd.Timedelta | None
        Порог, после которого разрыв считается паузой между файлами.
        Если None, берется 10 * медианный шаг.

    Returns
    -------
    float
    """
    timestamps = pd.DatetimeIndex(pd.to_datetime(timestamps)).sort_values()

    if len(timestamps) < 2:
        return 0.0

    diffs = timestamps.to_series().diff().dropna()

    if gap_threshold is None:
        median_step = diffs.median()
        gap_threshold = 10 * median_step
    else:
        gap_threshold = pd.Timedelta(gap_threshold)

    covered = diffs[diffs <= gap_threshold].sum()

    duration_days = covered.total_seconds() / 86400

    if duration_days <= 0:
        return 0.0

    return len(pred_events) / duration_days

def add_results(
    df,
    model_name,
    pr_auc,
    f1_at_fpr_1,
    recall,
    precision,
    precision_at_100,
    recall_at_100,
    event_recall,
    event_precision,
    f1_event,
    ttd_mean,
    ttd_median,
    alerts,
    n_true_events,
    n_pred_events,
    train_time_sec,
    inference_time_sec,
    model_params,
    threshold,
    f1_anom
):
    '''
    Добавляет результаты модели в DataFrame.
    '''
    #safe_model_params = make_json_serializable(model_params)
    row = {
        'PR-AUC': round(pr_auc*100, 2),
        'F1@FPR5%': round(f1_at_fpr_1*100, 2),
        'Recall': round(recall*100, 2),
        'Precision': round(precision*100, 2),
        'Precision@100': round(precision_at_100*100, 2),
        'Recall@100': round(recall_at_100*100, 2),
        'Event Recall': round(event_recall*100, 2),
        'Event Precision': round(event_precision*100, 2),
        'F1 event': round(f1_event*100, 2),
        'TTD mean': round(ttd_mean, 2),
        'TTD median': round(ttd_median, 2),
        'Alerts per day': round(alerts, 2),
        'n true events': n_true_events,
        'n pred events': n_pred_events,
        'threshold': threshold,
        'train time sec': round(train_time_sec, 2),
        'inference time sec': round(inference_time_sec, 2),
        #'model params': json.dumps(safe_model_params, ensure_ascii=False),
        'F1_anom': round(f1_anom*100, 2)
    }

    if df.empty:
        df = pd.DataFrame(columns=row.keys())

    for col in row.keys():
        if col not in df.columns:
            df[col] = pd.NA

    df.loc[f'{model_name}'] = row
    return df

def make_json_serializable(obj):
    '''
    Приводит объект к JSON-сериализуемому виду.

    Поддерживает базовые типы, numpy, списки, словари,
    pandas Timestamp/Timedelta и рекурсивно обрабатывает вложенные структуры.

    Параметры:
        obj: произвольный объект

    Возвращает:
        JSON-совместимое представление объекта
    '''
    if obj is None:
        return None
    if isinstance(obj, (str, int, float, bool)):
        return obj
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (list, tuple)):
        return [make_json_serializable(x) for x in obj]
    if isinstance(obj, dict):
        return {str(k): make_json_serializable(v) for k, v in obj.items()}
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    if isinstance(obj, pd.Timedelta):
        return str(obj)

    # для объектов вроде Identity, sklearn/torch transforms и т.п.
    return str(obj)

def plot_pr_curve(y_true, y_scores, model_name='model'):
    '''
    Строит Precision-Recall кривую и возвращает PR-AUC.

    Параметры:
        y_true (array-like): бинарные метки (0/1)
        y_scores (array-like): anomaly score
        model_name (str): имя модели для легенды

    Возвращает:
        float: значение PR-AUC
    '''
    precision, recall, thresholds = precision_recall_curve(y_true, y_scores)
    pr_auc = average_precision_score(y_true, y_scores)

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=recall,
        y=precision,
        mode='lines',
        name=f'{model_name} (PR-AUC={pr_auc:.4f})'
    ))

    fig.update_layout(
        title='Precision-Recall Curve',
        xaxis_title='Recall',
        yaxis_title='Precision',
        template='plotly_white'
    )

    fig.show()

    return pr_auc

def rank_transform(scores, ascending=True):
    scores = pd.Series(scores)
    return scores.rank(method='average', pct=True, ascending=ascending)

def calibrate_scores_by_val(val_scores_series, test_scores_series, higher_is_more_anomalous=True):
    val_scores_series = pd.Series(val_scores_series).dropna().sort_values().reset_index(drop=True)
    test_scores_series = pd.Series(test_scores_series)

    val_array = val_scores_series.to_numpy()
    test_array = test_scores_series.to_numpy()

    if higher_is_more_anomalous:
        # доля val-значений <= test_score
        calibrated = np.searchsorted(val_array, test_array, side="right") / len(val_array)
    else:
        # доля val-значений >= test_score
        calibrated = 1.0 - (np.searchsorted(val_array, test_array, side="left") / len(val_array))

    return pd.Series(calibrated, index=test_scores_series.index)

In [ ]:
def patched_ae_forward(self, x, return_latent=False):
    enc = self.encoder(x.reshape(x.shape[0], -1).float())
    dec = self.decoder(enc)
    recon_x = dec.reshape(x.shape)
    if return_latent:
        return recon_x, enc
    return recon_x


def patched_dagmm_forward(self, x):
    dec, enc = self.autoencoder(x, return_latent=True)

    a = x.reshape(x.shape[0], -1)
    b = dec.reshape(dec.shape[0], -1)

    cos_distance = F.cosine_similarity(a, b, dim=1).unsqueeze(-1)
    euclidean_distance = ((a - b) ** 2).mean(dim=1).sqrt().unsqueeze(-1)

    z = torch.cat([enc, euclidean_distance, cos_distance], dim=1)
    gamma = self.estimation(z)

    return enc, dec, z, gamma


dagmm_module.AEModule.forward = patched_ae_forward
dagmm_module.DAGMMModule.forward = patched_dagmm_forward

In [ ]:
def load_split_data(num_data, use_miss=True):
    train = pd.read_csv(f'../data_post/{num_data}/train{num_data}.csv', index_col=0)
    val = pd.read_csv(f'../data_post/{num_data}/val{num_data}.csv', index_col=0)

    target_cols = None
    if not use_miss:
        target_cols = [c for c in train.columns if c == c.replace('miss_', '', 1)]
        train, val = train[target_cols], val[target_cols]
        target_cols.append('target')

    if num_data == 6:
        test1 = pd.read_csv(f'../data_post/{num_data}/test{num_data}1.csv', index_col=0)
        test2 = pd.read_csv(f'../data_post/{num_data}/test{num_data}2.csv', index_col=0)
        if not use_miss:
            test1 = test1[target_cols]
            test2 = test2[target_cols]
        return train, val, test1, test2

    test = pd.read_csv(f'../data_post/{num_data}/test{num_data}.csv', index_col=0)
    if not use_miss:
        test = test[target_cols]
    return train, val, test


def prepare_scaled_data(num_data, list_anomaly, scaler_cls, use_miss=True, use_scaling=True):
    loaded = load_split_data(num_data=num_data, use_miss=use_miss)
    train, val = loaded[0], loaded[1]

    min_len = event_lengths(list_anomaly)['min']
    median_len = event_lengths(list_anomaly)['median']
    gap_fill, min_event, event_interval, window_size = event_params(min_len, median_len)

    feature_names = list(train.columns)

    scaler = scaler_cls if use_scaling else None

    if use_scaling:
        train_scaled = scaler.fit_transform(train[feature_names])
        val_scaled = scaler.transform(val[feature_names])
        train_norm = pd.DataFrame(train_scaled, index=train.index, columns=feature_names)
        val_norm = pd.DataFrame(val_scaled, index=val.index, columns=feature_names)
    else:
        train_norm = train[feature_names].copy()
        val_norm = val[feature_names].copy()

    # преобразование в TimeSeries
    train_data = TimeSeries.from_pd(train_norm)
    val_data = TimeSeries.from_pd(val_norm)

    result = {
        "train": train,
        "val": val,
        "train_data": train_data,
        "val_data": val_data,
        "feature_names": feature_names,
        "gap_fill": gap_fill,
        "min_event": min_event,
        "event_interval": event_interval,
        "window_size": window_size,
        "scaler": scaler,
        "use_scaling": use_scaling,
    }

    if num_data == 6:
        test1, test2 = loaded[2], loaded[3]
        test = pd.concat([test1, test2]).sort_index()

        if use_scaling:
            test_scaled1 = scaler.transform(test1[feature_names])
            test_scaled2 = scaler.transform(test2[feature_names])
            test_norm1 = pd.DataFrame(test_scaled1, index=test1.index, columns=feature_names)
            test_norm2 = pd.DataFrame(test_scaled2, index=test2.index, columns=feature_names)
        else:
            test_norm1 = test1[feature_names].copy()
            test_norm2 = test2[feature_names].copy()

        result["test"] = test
        result["test_data1"] = TimeSeries.from_pd(test_norm1)
        result["test_data2"] = TimeSeries.from_pd(test_norm2)

        y_true = pd.concat([test1['target'], test2['target']]).sort_index()
        y_true.index = pd.to_datetime(y_true.index)
        result["y_true"] = y_true
    else:
        test = loaded[2]

        if use_scaling:
            test_scaled = scaler.transform(test[feature_names])
            test_norm = pd.DataFrame(test_scaled, index=test.index, columns=feature_names)
        else:
            test_norm = test[feature_names].copy()

        result["test"] = test
        result["test_data"] = TimeSeries.from_pd(test_norm)

        y_true = test['target'].astype(int)
        y_true.index = pd.to_datetime(y_true.index)
        result["y_true"] = y_true

    return result

def fit_model(model, models, config, prepared, num_data, model_name_save):
    set_seed(42)
    model_train = model(config, models)

    start_train = time.perf_counter()
    model_train.train(prepared["train_data"])
    train_time_sec = time.perf_counter() - start_train

    os.makedirs("../models", exist_ok=True)
    model_name = f'{model_name_save}{num_data}'
    model_train.save(f'../models/{model_name}')

    return model_train, train_time_sec

def score_model(model, prepared, num_data, model_name_save):

    os.makedirs("../models", exist_ok=True)
    path = os.path.join("../models", f"{model_name_save}{num_data}")
    model_train = model.load(dirname=path)

    start_infer = time.perf_counter()
    val_scores = model_train.get_anomaly_score(prepared["val_data"]).to_pd().iloc[:, 0]

    if num_data == 6:
        test_scores1 = model_train.get_anomaly_score(prepared["test_data1"]).to_pd().iloc[:, 0]
        test_scores2 = model_train.get_anomaly_score(prepared["test_data2"]).to_pd().iloc[:, 0]
        test_scores = pd.concat([test_scores1, test_scores2]).sort_index()
    else:
        test_scores = model_train.get_anomaly_score(prepared["test_data"]).to_pd().iloc[:, 0]

    inference_time_sec = time.perf_counter() - start_infer

    return model_train, val_scores, test_scores, inference_time_sec

def compute_metrics(y_true, test_scores, val_scores, q, alpha,
                    gap_fill, min_event, event_interval, num_data,
                    use_calibrate=False, higher_is_more_anomalous=True):
    test_scores = test_scores.copy()
    val_scores = val_scores.copy()

    if use_calibrate:
        val_scores = calibrate_scores_by_val(val_scores_series=val_scores, test_scores_series=test_scores, higher_is_more_anomalous=higher_is_more_anomalous)
        test_scores = calibrate_scores_by_val(val_scores_series=val_scores, test_scores_series=test_scores, higher_is_more_anomalous=higher_is_more_anomalous)

    test_scores.index = y_true.index

    threshold = float(val_scores.quantile(q) * alpha)
    y_pred = (test_scores > threshold).astype(int)
    y_pred_bin = pd.Series(y_pred.values, index=y_true.index)

    pred_events = binary_to_events(y_pred_bin, gap_minutes=gap_fill, min_len_minutes=min_event)
    true_events = binary_to_events(y_true, gap_minutes=gap_fill, min_len_minutes=1)

    event_r = event_recall(true_events, pred_events, max_min=event_interval)
    event_p = event_precision(true_events, pred_events, max_min=event_interval)
    f1_event = 0 if (event_p + event_r) == 0 else 2 * event_p * event_r / (event_p + event_r)

    alerts = alerts_per_day(pred_events, y_true.index)
    if num_data == 6:
        alerts *= 1.4

    return {
        "threshold_value": threshold,
        "y_pred": y_pred,
        "y_pred_bin": y_pred_bin,
        "pred_events": pred_events,
        "true_events": true_events,
        "pr_auc_val": pr_auc(y_true, test_scores),
        "f1_val": f1_at_fpr(y_true, test_scores, target_fpr=0.05),
        "p100": precision_at_k(y_true, test_scores, k=100),
        "r100": recall_at_k(y_true, test_scores, k=100),
        "recall": recall_score(y_true, y_pred_bin, pos_label=1),
        "precision": precision_score(y_true, y_pred_bin, pos_label=1),
        "f1_anom": f1_score(y_true, y_pred, pos_label=1),
        "event_r": event_r,
        "event_p": event_p,
        "f1_event": f1_event,
        "ttd_list": time_to_detect(true_events, pred_events, reduction='list', unit='minutes', max_min=event_interval),
        "ttd_mean": time_to_detect(true_events, pred_events, reduction='mean', unit='minutes', max_min=event_interval),
        "ttd_median": time_to_detect(true_events, pred_events, reduction='median', unit='minutes', max_min=event_interval),
        "alerts": alerts,
        "cm": confusion_matrix(y_true, y_pred),
        "test_scores": test_scores,
        "val_scores": val_scores,
    }


def save_results_sheet(num_data, model_name, metrics, model_params,
                       train_time_sec, inference_time_sec, val_scores, q, alpha):
    file_path = '../results.xlsx'
    sheet_name = str(num_data)

    if os.path.exists(file_path):
        try:
            results = pd.read_excel(file_path, sheet_name=sheet_name, index_col=0)
        except ValueError:
            results = pd.DataFrame()
    else:
        results = pd.DataFrame()

    # если время не передано, пробуем взять старое из таблицы
    if model_name in results.index:
        if train_time_sec is None and 'train time sec' in results.columns:
            old_val = results.loc[model_name, 'train time sec']
            if pd.notna(old_val):
                train_time_sec = float(old_val)

        if inference_time_sec is None and 'inference time sec' in results.columns:
            old_val = results.loc[model_name, 'inference time sec']
            if pd.notna(old_val):
                inference_time_sec = float(old_val)

        if model_params is None and 'model params' in results.columns:
            old_params = results.loc[model_name, 'model params']
            if pd.notna(old_params):
                model_params = old_params


    results = add_results(
        df=results,
        model_name=model_name,
        pr_auc=metrics["pr_auc_val"],
        f1_at_fpr_1=metrics["f1_val"],
        recall=metrics["recall"],
        precision=metrics["precision"],
        precision_at_100=metrics["p100"],
        recall_at_100=metrics["r100"],
        event_recall=metrics["event_r"],
        event_precision=metrics["event_p"],
        f1_event=metrics["f1_event"],
        ttd_mean=metrics["ttd_mean"],
        ttd_median=metrics["ttd_median"],
        alerts=metrics["alerts"],
        n_true_events=len(metrics["true_events"]),
        n_pred_events=len(metrics["pred_events"]),
        train_time_sec=train_time_sec,
        inference_time_sec=inference_time_sec,
        model_params=model_params,
        threshold=f'{q}, {"-" if val_scores.quantile(q) < 0 else "+"}, {alpha}',
        f1_anom=metrics["f1_anom"]
    )

    mode = 'a' if os.path.exists(file_path) else 'w'
    if mode == 'a':
        with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
            results.to_excel(writer, sheet_name=sheet_name, index=True)
    else:
        with pd.ExcelWriter(file_path, engine='openpyxl', mode='w') as writer:
            results.to_excel(writer, sheet_name=sheet_name, index=True)

    wb = load_workbook(file_path)
    ws = wb[sheet_name]
    ws.column_dimensions['A'].width = 40
    for i, col in enumerate(results.columns, start=2):
        ws.column_dimensions[get_column_letter(i)].width = max(
            results[col].astype(str).map(len).max(),
            len(col)
        ) + 2
    wb.save(file_path)
    wb.close()


def save_summary_sheet(num_data, model_name, metrics):
    file_path = '../results_all.xlsx'
    dataset_map = {
        2: '2: 96/193 параметров, 4800 минут',
        7: '7: 27/59 параметров, 5760 минут, X',
        3: '3: 15/31 параметров, 7920 минут, X',
        8: '8: 13/34 параметров, 22495 минут, X',
        6: '6: 3/7 параметров, 24192 минут, X',
    }
    dataset_name = dataset_map[num_data]

    sheet_to_metrics = {
        'PR_event': {
            'Event Recall': round(metrics["event_r"] * 100, 2),
            'Event Precision': round(metrics["event_p"] * 100, 2),
        },
        'PR-AUC, F1': {
            'F1 event': round(metrics["f1_event"] * 100, 2),
            'PR-AUC': round(metrics["pr_auc_val"] * 100, 2),
        }
    }

    for summary_sheet, metric_dict in sheet_to_metrics.items():
        if os.path.exists(file_path):
            try:
                summary_df = pd.read_excel(file_path, sheet_name=summary_sheet, header=[0, 1], index_col=0)
            except ValueError:
                summary_df = pd.DataFrame()
        else:
            summary_df = pd.DataFrame()

        for metric_name in metric_dict.keys():
            col = (dataset_name, metric_name)
            if col not in summary_df.columns:
                summary_df[col] = pd.NA

        for metric_name, metric_value in metric_dict.items():
            summary_df.loc[model_name, (dataset_name, metric_name)] = metric_value

        mode = 'a' if os.path.exists(file_path) else 'w'
        if mode == 'a':
            with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
                summary_df.to_excel(writer, sheet_name=summary_sheet)
        else:
            with pd.ExcelWriter(file_path, engine='openpyxl', mode='w') as writer:
                summary_df.to_excel(writer, sheet_name=summary_sheet)

    wb = load_workbook(file_path)
    for summary_sheet in sheet_to_metrics.keys():
        ws = wb[summary_sheet]
        ws.column_dimensions['A'].width = 40
        for col_idx in range(2, ws.max_column + 1):
            ws.column_dimensions[get_column_letter(col_idx)].width = 20
        ws.freeze_panes = "B3"
    wb.save(file_path)
    wb.close()


def print_metrics(metrics):
    print('PR-AUC:', metrics["pr_auc_val"])
    print('Recall:', metrics["recall"])
    print('Precision:', metrics["precision"])
    print('F1_anom', metrics["f1_anom"])
    print('F1 @ FPR=5%:', metrics["f1_val"])
    print('Recall@100:', metrics["r100"])
    print('Precision@100:', metrics["p100"])
    print('Event Recall:', metrics["event_r"])
    print('Event Precision:', metrics["event_p"])
    print('F1 event:', metrics["f1_event"])
    print('Time-to-detect mean (minutes):', metrics["ttd_mean"])
    print('Time-to-detect median (minutes):', metrics["ttd_median"])
    print('Time-to-detect list (minutes):', metrics["ttd_list"])
    print('Alerts per day:', metrics["alerts"])
    print('n true events:', len(metrics["true_events"]))
    print('n pred events:', len(metrics["pred_events"]))
    print(metrics["cm"])


def train_func(num_data, config, list_anomaly, model_name_save, model_name, models, q=0.95, alpha=1,
               model=None, train_time_sec=None, inference_time_sec=None,
               val_scores=None, test_score_outside=None, save_result=True,
               no_train=True, use_miss=True, use_calibrate=False, higher_is_more_anomalous=True, use_scaling=True, scaler_cls=RobustScaler()):
    prepared = prepare_scaled_data(num_data=num_data, list_anomaly=list_anomaly, use_miss=use_miss, use_scaling=use_scaling, scaler_cls=scaler_cls)

    print(f'Размерности: train: {prepared["train"].shape}, val: {prepared["val"].shape}')
    print(
        f'Параметры исследования аномалий: максимальный пропуск: {prepared["gap_fill"]}, '
        f'минимальная длина аномалии: {prepared["min_event"]}, '
        f'интервал вокруг начала аномалии: {prepared["event_interval"]}, '
        f'размер окна: {prepared["window_size"]}'
    )
    print(f'Количество аномальных точек в датасете: {prepared["y_true"].sum()}')

    models[1].config.sequence_len = prepared["window_size"]
    models[2].config.sequence_len = prepared["window_size"]

    if not no_train:
        model_train, train_time_sec = fit_model(
            model=model,
            models=models,
            config=config,
            prepared=prepared,
            num_data=num_data,
            model_name_save=model_name_save
        )

        start_infer = time.perf_counter()
        val_scores = model_train.get_anomaly_score(prepared["val_data"]).to_pd().iloc[:, 0]

        if num_data == 6:
            test_scores1 = model_train.get_anomaly_score(prepared["test_data1"]).to_pd().iloc[:, 0]
            test_scores2 = model_train.get_anomaly_score(prepared["test_data2"]).to_pd().iloc[:, 0]
            test_scores = pd.concat([test_scores1, test_scores2])
        else:
            test_scores = model_train.get_anomaly_score(prepared["test_data"]).to_pd().iloc[:, 0]

        inference_time_sec = time.perf_counter() - start_infer

    else:
        if val_scores is None and test_score_outside is None:
            model_train, val_scores, test_scores, inference_time_sec = score_model(
                model=model,
                prepared=prepared,
                num_data=num_data,
                model_name_save=model_name_save
            )
        else:
            model_train = None
            train_time_sec = None
            inference_time_sec = None

            if val_scores is None:
                raise ValueError("При no_train=True нужно передать val_scores или загрузить модель")
            if test_score_outside is None:
                raise ValueError("При no_train=True нужно передать test_score_outside или загрузить модель")

            test_scores = test_score_outside

    metrics = compute_metrics(
        y_true=prepared["y_true"],
        test_scores=test_scores,
        val_scores=val_scores,
        q=q,
        alpha=alpha,
        gap_fill=prepared["gap_fill"],
        min_event=prepared["min_event"],
        event_interval=prepared["event_interval"],
        num_data=num_data,
        use_calibrate=use_calibrate,
        higher_is_more_anomalous=higher_is_more_anomalous,
    )

    print_metrics(metrics)

    pred_events = metrics["pred_events"]
    true_events = metrics["true_events"]

    if num_data == 6 and len(pred_events) <= 100:
        fig = make_subplots(
            rows=2, cols=1, shared_xaxes=True,
            row_heights=[0.3, 0.7], vertical_spacing=0.05,
            specs=[[{'secondary_y': True}], [{}]]
        )

        for start, end in true_events:
            fig.add_trace(
                go.Scatter(
                    x=[start, end],
                    y=[1, 1],
                    mode='lines',
                    line=dict(width=12, color='green'),
                    showlegend=False,
                    name='true_event'
                ),
                row=1, col=1,
                secondary_y=False
            )
            fig.add_vline(
                x=start,
                line_width=2,
                line_dash='dash',
                line_color='green',
                opacity=0.7,
                row=1, col=1
            )

        for start, end in pred_events:
            fig.add_trace(
                go.Scatter(
                    x=[start, end],
                    y=[0, 0],
                    mode='lines',
                    line=dict(width=12, color='red'),
                    showlegend=False,
                    name='pred_event'
                ),
                row=1, col=1,
                secondary_y=False
            )
            fig.add_vline(
                x=start,
                line_width=2,
                line_dash='dot',
                line_color='red',
                opacity=0.7,
                row=1, col=1
            )

        fig.add_trace(
            go.Scatter(
                x=metrics["test_scores"].index,
                y=metrics["test_scores"],
                line=dict(color='black', width=2),
                name='score'
            ),
            row=1, col=1,
            secondary_y=True
        )

        df = prepared["test"].copy()
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df.iloc[:, 0],
                line=dict(color='blue'),
                name=df.columns[0]
            ),
            row=2, col=1
        )
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df.iloc[:, 1],
                line=dict(color='orange'),
                name=df.columns[1]
            ),
            row=2, col=1
        )

        fig.update_layout(
            template='plotly_white',
            height=600,
            title='Signals + Events + Score'
        )
        fig.update_yaxes(
            tickvals=[0, 1],
            ticktext=['pred', 'true'],
            row=1, col=1
        )
        fig.show()

    if len(pred_events) <= 100:
        fig = go.Figure()

        for start, end in true_events:
            fig.add_trace(
                go.Scatter(
                    x=[start, end],
                    y=[1, 1],
                    mode='lines',
                    line=dict(width=12, color='green'),
                    name='true_event',
                    showlegend=False
                )
            )
            fig.add_vline(
                x=start,
                line_width=2,
                line_dash='dash',
                line_color='green',
                opacity=0.7
            )

        for start, end in pred_events:
            fig.add_trace(
                go.Scatter(
                    x=[start, end],
                    y=[0, 0],
                    mode='lines',
                    line=dict(width=12, color='red'),
                    name='pred_event',
                    showlegend=False
                )
            )
            fig.add_vline(
                x=start,
                line_width=2,
                line_dash='dot',
                line_color='red',
                opacity=0.7
            )

        fig.update_layout(
            yaxis=dict(
                tickvals=[0, 1],
                ticktext=['pred', 'true']
            ),
            title='True vs Predicted Events',
            template='plotly_white',
            height=300
        )
        fig.show()

    if save_result:
        save_results_sheet(
            num_data=num_data,
            model_name=model_name,
            metrics=metrics,
            model_params=model_train.config.__dict__,
            train_time_sec=train_time_sec,
            inference_time_sec=inference_time_sec,
            val_scores=val_scores,
            q=q,
            alpha=alpha,
        )
        save_summary_sheet(
            num_data=num_data,
            model_name=model_name,
            metrics=metrics,
        )

    return model_train, val_scores, test_scores, train_time_sec, inference_time_sec

In [ ]:
list2 = [['2026-01-19 13:31', '2026-01-19 14:44'], ['2026-01-20 09:02', '2026-01-20 11:29'], ['2026-01-23 08:54', '2026-01-23 10:15']]
list3 = [['2026-02-05 09:00', '2026-02-05 10:00'], ['2026-02-05 18:00', '2026-02-05 20:00'], ['2026-02-06 23:00', '2026-02-09 14:00'],
        ['2026-02-16 13:00', '2026-02-16 15:00']
]
list4 = [['2026-02-08 13:42', '2026-02-08 14:11'], ['2026-02-10 15:46', '2026-02-10 17:12'], ['2026-02-12 07:04', '2026-02-12 08:25']]
list6 = [['2025-12-13 17:23', '2025-12-13 17:30'], ['2025-12-13 18:41', '2025-12-13 18:44'], ['2025-12-19 12:19', '2025-12-19 12:40'],
         ['2025-12-22 18:59', '2025-12-22 19:00'], ['2026-01-24 22:01', '2026-01-24 22:02'], ['2026-01-31 22:09', '2026-01-31 23:57'],
         ['2026-02-04 16:47', '2026-02-04 16:54'], ['2026-02-04 17:14', '2026-02-04 17:28']
]
list7 = [['2026-03-03 09:15', '2026-03-03 09:30'], ['2026-03-03 13:10', '2026-03-03 13:20'], ['2026-03-04 13:00', '2026-03-04 13:20']]
list8 = [['2026-02-23 09:44', '2026-02-23 13:19'], ['2026-02-24 13:26', '2026-02-24 13:42'], ['2026-02-24 13 17:36', '2026-02-24 19:36']]

list_anomaly = [2, 3, 4, 6]

In [ ]:
from merlion.models.anomaly.isolation_forest import IsolationForest, IsolationForestConfig
from merlion.models.anomaly.dagmm import DAGMM, DAGMMConfig
from merlion.models.anomaly.lstm_ed import LSTMED, LSTMEDConfig

from merlion.models.ensemble.anomaly import DetectorEnsemble, DetectorEnsembleConfig
from merlion.models.ensemble.combine import Mean, Median

q = 0.95
no_train = True
save_result = True
use_miss = True
use_scaling = True
use_rank_transform = False
ascending = True
enable_calibrator = False
enable_threshold = False
model = DetectorEnsemble
model_name_result = 'Ensemble'
model_name = ''

model_if = IsolationForest(
    IsolationForestConfig(
        n_estimators=200,
        max_n_samples='auto',
        enable_threshold=enable_threshold, # нужны raw scores
        enable_calibrator=enable_calibrator,   # удобно для сопоставимых скоров
        transform=Identity(), normalize=Identity()
    )
)

model_dagmm = DAGMM(
    DAGMMConfig(
        batch_size=64,
        hidden_size=32,
        gmm_k=6,
        lr=1e-3,
        lambda_energy=0.03,
        lambda_cov_diag=0.001,
        num_epochs=20,
        enable_threshold=enable_threshold,
        enable_calibrator=enable_calibrator,
        transform=Identity(), normalize=Identity()
    )
)

model_lstm = LSTMED(
    LSTMEDConfig(
        n_layers=[3, 3],
        dropout=[0.2, 0.2],
        hidden_size=16,
        batch_size=64,
        num_epochs=20,
        enable_threshold=enable_threshold,
        enable_calibrator=enable_calibrator,
        transform=Identity(), normalize=Identity()
    )
)

model_ae = AutoEncoder(
    AutoEncoderConfig(
        hidden_size=16,
        layer_sizes=(64, 32, 16),
        num_epochs=80,
        batch_size=128,
        lr=1e-3,
        enable_threshold=enable_threshold,
        enable_calibrator=enable_calibrator,
        transform=Identity(), normalize=Identity()
    )
)
config = DetectorEnsembleConfig(
        combiner=Mean(abs_score=True),   # можно заменить на Median(abs_score=True)
        enable_threshold=enable_threshold,          # на выходе raw ensemble scores
        enable_calibrator=enable_calibrator,
        transform=Identity(), normalize=Identity()
    )

models = [model_if, model_dagmm, model_lstm]

In [ ]:
for num in [8]:
    model = DetectorEnsemble
    model_name_result = 'Ensemble_Merlion'
    globals()[f'model{num}'], globals()[f'val_score{num}'], globals()[f'test_score{num}'], globals()[f'train_time{num}'], globals()[f'inference_time{num}'] = train_func(
                                        model=model,
                                        models=models,
                                        config=config,
                                        num_data=num, 
                                        list_anomaly=globals()[f'list{num}'],
                                        model_name_result=model_name_result,
                                        model_name=model_name,
                                        q = q,
                                        no_train=no_train,
                                        save_result=save_result,
                                        use_miss=use_miss,
                                        
)

Размерности: train: (22495, 34), val: (5624, 34)
Параметры исследования аномалий: максимальный пропуск: 15, минимальная длина аномалии: 8, интервал вокруг начала аномалии: 96, размер окна: 48
Количество аномальных точек в датасете: 354
 |========================================| 100.0% Complete, Loss 2.2242, Recon_error: 2.55051
 |========================================| 100.0% Complete, Loss 95432.4123492
PR-AUC: 0.0419808616702647
Recall: 0.768361581920904
Precision: 0.06217142857142857
F1_anom 0.11503489109748362
F1 @ FPR=5%: 0.008771929824561403
Recall@100: 0.002824858757062147
Precision@100: 0.01
Event Recall: 0.6666666666666666
Event Precision: 0.05714285714285714
F1 event: 0.10526315789473684
Time-to-detect mean (minutes): 48.0
Time-to-detect median (minutes): 48.0
Time-to-detect list (minutes): [-58.0, 38.0]
Alerts per day: 7.009735744089013
n true events: 3
n pred events: 35
[[2734 4103]
 [  82  272]]


### other

In [42]:
model6, val_score6, test_score6, train_time6, inference_time6 = train_func(
                                        model=model,
                                        config=config,
                                        models = models,
                                        num_data=6, 
                                        list_anomaly=list6,
                                        model_name_result=model_name_result,
                                        use_miss=use_miss,
                                        no_train=False
)

Размерности: train: (24192, 7), val: (6048, 7)
Параметры исследования аномалий: максимальный пропуск: 1, минимальная длина аномалии: 3, интервал вокруг начала аномалии: 20, размер окна: 5


Caught an exception while training model 1/3 (IsolationForest). Model will not be used. Traceback (most recent call last):
  File "c:\Users\al.melnikova\AppData\Local\anaconda3\envs\py311cuda\Lib\site-packages\merlion\models\ensemble\anomaly.py", line 162, in _train
    train_scores, valid_scores = TSADEvaluator(model=model, config=eval_cfg).get_predict(
                                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\al.melnikova\AppData\Local\anaconda3\envs\py311cuda\Lib\site-packages\merlion\evaluate\anomaly.py", line 443, in get_predict
    train_result, result = super().get_predict(
                           ^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\al.melnikova\AppData\Local\anaconda3\envs\py311cuda\Lib\site-packages\merlion\evaluate\base.py", line 202, in get_predict
    train_result = self._train_model(train_vals, **full_train_kwargs)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\al.melnikova\AppDat

Количество аномальных точек в датасете: 170
 |========================================| 100.0% Complete, Loss -0.3210, Recon_error: 0.1066


Obtained max score of 51141562368.00, but self.max_score is only 1000.00. Updating self.max_score to 102283124736.00.


 |========================================| 100.0% Complete, Loss 537.013683


Unused kwargs: {'sequence_len': 5}
Stack (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\al.melnikova\AppData\Local\anaconda3\envs\py311cuda\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\al.melnikova\AppData\Local\anaconda3\envs\py311cuda\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\al.melnikova\AppData\Local\anaconda3\envs\py311cuda\Lib\site-packages\ipykernel\kernelapp.py", line 758, in start
    self.io_loop.start()
  File "c:\Users\al.melnikova\AppData\Local\anaconda3\envs\py311cuda\Lib\site-packages\tornado\platform\asyncio.py", line 211, in start
    self.asyncio_loop.run_forever()
  File "c:\Users\al.melnikova\AppData\Local\anaconda3\envs\py311cuda\Lib\asyncio\base_events.py", line 608, in run_forever
    self._run_once()
  File "c:\Users\al.melnikov

PR-AUC: 0.0029314993680662595
PR-AUC: 0.0029314993680662595
Recall: 0.08235294117647059
Precision: 0.0015990862364363221
F1_anom 0.003137254901960784
F1 @ FPR=5%: 0.0069221260815822
Precision@100: 0.0
Recall@100: 0.0
Event Recall: 0.25
Event Precision: 0.002824858757062147
F1 event: 0.00558659217877095
Time-to-detect mean (minutes): 10.0
Time-to-detect median (minutes): 10.0
Time-to-detect list (minutes): [7.0, 13.0]
Alerts per day: 13.042343610080593
n true events: 8
n pred events: 708
[[70288  8741]
 [  156    14]]


In [35]:
train_func(
      config=None,
      model=model,
      models=models,
      num_data=6,
      list_anomaly=list6,
      model_name_result=model_name_result,
      q = 0.95,
      alpha=1,
      save_result=False,
      train_time_sec=train_time6,
      inference_time_sec=inference_time6,
      val_score=val_score6,
      test_score_outside=test_score6,
      use_miss=use_miss
)

Размерности: train: (24192, 7), val: (6048, 7)
Параметры исследования аномалий: максимальный пропуск: 1, минимальная длина аномалии: 3, интервал вокруг начала аномалии: 20, размер окна: 5
Количество аномальных точек в датасете: 170
PR-AUC: 0.12098862788842189
Recall: 0.9176470588235294
Precision: 0.014464534075104311
F1_anom 0.028480146052031037
F1 @ FPR=5%: 0.06666666666666667
Precision@100: 0.2
Recall@100: 0.11764705882352941
Event Recall: 0.75
Event Precision: 0.01098901098901099
F1 event: 0.021660649819494587
Time-to-detect mean (minutes): 1.0
Time-to-detect median (minutes): 1.0
Time-to-detect list (minutes): [1.0, 1.0, 1.0, 0.0, 1.0, 2.0]
Alerts per day: 10.058078546757066
n true events: 8
n pred events: 546
[[68400 10629]
 [   14   156]]


### обучение

In [ ]:
model2, val_score2, test_score2, train_time2, inference_time2 = train_func(
                                        model=model,
                                        config=config,
                                        models=models,
                                        num_data=2, 
                                        list_anomaly=list2,
                                        model_name_result=model_name_result,
                                        use_miss=use_miss,
                                        no_train=False
)

In [ ]:
model3, val_score3, test_score3, train_time3, inference_time3 = train_func(
                                        model=model,
                                        config = config,
                                        num_data=3,
                                        list_anomaly=list3,
                                        model_name_result=model_name_result,
                                        use_miss=use_miss,
                                        no_train=False
                                  )

In [ ]:
model6, val_score6, test_score6, train_time6, inference_time6 = train_func(
                                        config=config,
                                        model=model,
                                        num_data=6,
                                        list_anomaly=list6,
                                        model_name_result=model_name_result,
                                        use_miss=use_miss,
                                        no_train=False
                                  )

In [ ]:
model7, val_score7, test_score7, train_time7, inference_time7 = train_func(
                                        config=config,
                                        model=model,
                                        num_data=7,
                                        list_anomaly=list7,
                                        model_name_result=model_name_result,
                                        use_miss=use_miss,
                                        no_train=False
                                  )

In [ ]:
model8, val_score8, test_score8, train_time8, inference_time8 = train_func(
                                        config=config,
                                        model=model,
                                        num_data=8,
                                        list_anomaly=list8,
                                        model_name_result=model_name_result,
                                        use_miss=use_miss,
                                        no_train=False
                                  )

### подбор параметров

In [ ]:
train_func(
      config=config,
      model=model2,
      num_data=2,
      list_anomaly=list2,
      model_name_result=model_name_result,
      q = 0.9,
      save_result=True,
      train_time_sec=train_time2,
      inference_time_sec=inference_time2,
      val_score=val_score2,
      test_score_outside=test_score2,
      use_miss=use_miss
)

In [ ]:
train_func(
      config=config,
      model=model3,
      num_data=3,
      list_anomaly=list3,
      model_name_result=model_name_result,
      q = 0.999,
      alpha=1.3,
      save_result=True,
      train_time_sec=train_time3,
      inference_time_sec=inference_time3,
      val_score=val_score3,
      test_score_outside=test_score3,
      use_miss=use_miss
)

In [22]:
train_func(
      config=config,
      model=model,
      models=models,
      num_data=6,
      list_anomaly=list6,
      model_name_result=model_name_result,
      q = 0.96,
      alpha=1,
      save_result=False,
      train_time_sec=train_time6,
      inference_time_sec=inference_time6,
      val_score=val_score6,
      test_score_outside=test_score6,
      use_miss=use_miss
)

Размерности: train: (24192, 7), val: (6048, 7)
Параметры исследования аномалий: максимальный пропуск: 1, минимальная длина аномалии: 3, интервал вокруг начала аномалии: 20, размер окна: 5
Количество аномальных точек в датасете: 170


AttributeError: type object 'DetectorEnsemble' has no attribute 'config'

In [ ]:
train_func(
      config=config,
      model=model7,
      num_data=7,
      list_anomaly=list7,
      model_name_result=model_name_result,
      q = 0.99,
      save_result=True,
      train_time_sec=train_time7,
      inference_time_sec=inference_time7,
      val_score=val_score7,
      test_score_outside=test_score7,
      use_miss=use_miss
)

In [ ]:
train_func(
      config=config,
      model=model8,
      num_data=8,
      list_anomaly=list8,
      model_name_result=model_name_result,
      q = 0.999,
      alpha=1,
      save_result=True,
      train_time_sec=train_time8,
      inference_time_sec=inference_time8,
      val_score=val_score8,
      test_score_outside=test_score8,
      use_miss=use_miss
)